In [1]:
import os
import numpy as np

INPUT_DIR = "velocity_score_kl_analysis"

FLOWS = [
    "linear",
    "operator",
    "trigonometric",
    "spherical"
]

for flow in FLOWS:

    print("\n" + "=" * 75)
    print(f"FLOW: {flow}")
    print("=" * 75)

    for pair in ["pair_01", "pair_02", "pair_19"]:

        path = os.path.join(
            INPUT_DIR,
            flow,
            pair,
            "velocity_score_kl_analysis.npz"
        )

        if not os.path.exists(path):
            print(f"\nMissing: {path}")
            continue

        data = np.load(path)

        print(f"\n{pair}")

        for key in data.files:
            arr = data[key]

            if arr.ndim == 0:
                print(f"  {key:30s} = {arr}")
            else:
                print(
                    f"  {key:30s} "
                    f"shape={arr.shape}"
                )


FLOW: linear

pair_01
  time                           shape=(201,)
  kl                             shape=(201,)
  theoretical_derivative         shape=(201,)
  finite_difference_kl           shape=(201,)
  temporal_flux_derivative       shape=(201,)
  mean_x_contribution            shape=(201,)
  mean_y_contribution            shape=(201,)
  mean_absolute_integrand        shape=(201,)
  std_integrand                  shape=(201,)
  bandwidth                      shape=(2,)
  x_axis                         shape=(128,)
  y_axis                         shape=(128,)
  psi_phi_dimensions             shape=(2,)
  real_data_mean                 shape=(2,)
  real_data_std                  shape=(2,)
  p_ref                          shape=(128, 128)
  p_t                            shape=(201, 128, 128)
  temporal_score_grid            shape=(201, 128, 128)

pair_02
  time                           shape=(201,)
  kl                             shape=(201,)
  theoretical_derivative         s

In [4]:
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ============================================================
# CONFIGURATION
# ============================================================

INPUT_DIR = "velocity_score_kl_analysis"
OUTPUT_DIR = "velocity_score_plots"

FLOWS = [
    "linear",
    "trigonometric",
    "spherical",
    "operator"
]

N_PAIRS = 19
DPI = 600

os.makedirs(OUTPUT_DIR, exist_ok=True)

# White publication plotting style
plt.style.use("default")

COLORS = {
    "linear": "#0072B2",
    "trigonometric": "#E69F00",
    "spherical": "#009E73",
    "operator": "#D55E00"
}

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "font.weight": "bold",

    "axes.labelsize": 14,
    "axes.labelweight": "bold",
    "axes.titlesize": 15,
    "axes.titleweight": "bold",

    "legend.fontsize": 11,

    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",

    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",

    "xtick.color": "black",
    "ytick.color": "black",

    "axes.linewidth": 1.8,
    "xtick.major.width": 1.8,
    "ytick.major.width": 1.8,
    "xtick.major.size": 6,
    "ytick.major.size": 6,

    "axes.grid": False,
    "lines.linewidth": 2.5,

    "savefig.dpi": 400,
    "savefig.bbox": "tight"
})
# ============================================================
# LOAD ALL PAIRS
# ============================================================

DATA = {}

for flow in FLOWS:

    DATA[flow] = {}

    for pair_idx in range(1, N_PAIRS + 1):

        pair = f"pair_{pair_idx:02d}"

        path = os.path.join(
            INPUT_DIR,
            flow,
            pair,
            "velocity_score_kl_analysis.npz"
        )

        if not os.path.exists(path):
            continue

        with np.load(path) as npz:
            DATA[flow][pair] = {
                key: npz[key].copy()
                for key in npz.files
            }

    print(
        f"{flow}: loaded "
        f"{len(DATA[flow])} pairs"
    )


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_pair_arrays(flow, key):

    arrays = []

    for pair_idx in range(1, N_PAIRS + 1):

        pair = f"pair_{pair_idx:02d}"

        if pair not in DATA[flow]:
            continue

        if key not in DATA[flow][pair]:
            continue

        arrays.append(DATA[flow][pair][key])

    return arrays


def get_mean_std(flow, key):

    arrays = get_pair_arrays(flow, key)

    if not arrays:
        return None, None, None

    times = DATA[flow][
        next(iter(DATA[flow]))
    ]["time"]

    valid = [
        arr for arr in arrays
        if arr.shape == times.shape
    ]

    if not valid:
        return None, None, None

    matrix = np.stack(valid, axis=0)

    mean = np.nanmean(matrix, axis=0)
    std = np.nanstd(matrix, axis=0)

    return times, mean, std


def save_figure(fig, name):

    fig.savefig(
        os.path.join(OUTPUT_DIR, name + ".png"),
        dpi=DPI,
        bbox_inches="tight"
    )

    fig.savefig(
        os.path.join(OUTPUT_DIR, name + ".pdf"),
        bbox_inches="tight"
    )

    plt.close(fig)


# ============================================================
# FIGURE 1: VELOCITY-SCORE CONTRIBUTION
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6))

for flow in FLOWS:

    times, mean, std = get_mean_std(
        flow,
        "theoretical_derivative"
    )

    if mean is None:
        continue

    ax.plot(
        times,
        mean,
        color=COLORS[flow],
        linewidth=2.5,
        label=flow.capitalize()
    )


ax.axhline(0, color="white", linestyle="--", alpha=0.6)

ax.set_xlabel("Time")
ax.set_ylabel(r"$\mathbb{E}_{p_t}[\mathbf{v}_t \cdot \Delta \mathbf{s}_t]$")
ax.set_title("Velocity–Score Contribution vs Time")

ax.legend(frameon=False)
fig.tight_layout()

save_figure(fig, "01_velocity_score_contribution")


# ============================================================
# FIGURE 2: PSI AND PHI CONTRIBUTIONS
# ============================================================

fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 5),
    sharex=True
)

components = [
    ("mean_x_contribution", r"$\psi$ contribution"),
    ("mean_y_contribution", r"$\phi$ contribution")
]

for ax, (key, title) in zip(axes, components):

    for flow in FLOWS:

        times, mean, std = get_mean_std(flow, key)

        if mean is None:
            continue

        ax.plot(
            times,
            mean,
            color=COLORS[flow],
            linewidth=2.2,
            label=flow.capitalize()
        )

    ax.axhline(0, color="white", linestyle="--", alpha=0.6)

    ax.set_title(title)
    ax.set_xlabel("Time")
    ax.set_ylabel("Contribution")

axes[0].legend(frameon=False)

fig.suptitle(
    "Coordinate-wise Velocity–Score Contributions",
    fontsize=15
)

fig.tight_layout()

save_figure(fig, "02_coordinate_contributions")


# ============================================================
# FIGURE 3: THEORETICAL VS FINITE-DIFFERENCE DERIVATIVE
# ============================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 10),
    sharex=True
)

axes = axes.flatten()

for ax, flow in zip(axes, FLOWS):

    times, theoretical, _ = get_mean_std(
        flow,
        "theoretical_derivative"
    )

    _, finite_diff, _ = get_mean_std(
        flow,
        "finite_difference_kl"
    )

    if theoretical is None or finite_diff is None:
        continue

    ax.plot(
        times,
        theoretical,
        linewidth=2.2,
        color="#4FC3F7",
        label="Velocity–score"
    )

    ax.plot(
        times,
        finite_diff,
        linewidth=2,
        color="#FFB74D",
        linestyle="--",
        label="Finite difference"
    )

    ax.axhline(
        0,
        color="white",
        linestyle=":",
        alpha=0.5
    )

    ax.set_title(flow.capitalize())
    ax.set_xlabel("Time")
    ax.set_ylabel(r"$dD_{\mathrm{KL}}/dt$")
    ax.legend(frameon=False)

fig.suptitle(
    "Theoretical and Numerical KL Derivatives",
    fontsize=16
)

fig.tight_layout()

save_figure(fig, "03_derivative_comparison")


# ============================================================
# FIGURE 4: VELOCITY-SCORE CONTRIBUTION HEATMAP
# ============================================================

for flow in FLOWS:

    rows = []
    pair_labels = []

    for pair_idx in range(1, N_PAIRS + 1):

        pair = f"pair_{pair_idx:02d}"

        if pair not in DATA[flow]:
            continue

        arr = DATA[flow][pair]["theoretical_derivative"]

        rows.append(arr)
        pair_labels.append(pair)

    if not rows:
        continue

    matrix = np.stack(rows, axis=0)

    times = DATA[flow][pair_labels[0]]["time"]

    fig, ax = plt.subplots(figsize=(13, 7))

    vmax = np.nanpercentile(np.abs(matrix), 98)

    im = ax.imshow(
        matrix,
        aspect="auto",
        origin="lower",
        extent=[
            times[0],
            times[-1],
            0.5,
            len(pair_labels) + 0.5
        ],
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax
    )

    ax.set_yticks(np.arange(1, len(pair_labels) + 1))
    ax.set_yticklabels(pair_labels)

    ax.set_xlabel("Time")
    ax.set_ylabel("ψ–φ Pair")

    ax.set_title(
        f"{flow.capitalize()}: Velocity–Score Contribution"
    )

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(r"$\mathbb{E}[\mathbf{v}\cdot\Delta\mathbf{s}]$")

    fig.tight_layout()

    save_figure(
        fig,
        f"04_heatmap_{flow}"
    )


# ============================================================
# FIGURE 5: MEAN ABSOLUTE INTEGRAND
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6))

for flow in FLOWS:

    times, mean, std = get_mean_std(
        flow,
        "mean_absolute_integrand"
    )

    if mean is None:
        continue

    ax.plot(
        times,
        mean,
        color=COLORS[flow],
        linewidth=2.3,
        label=flow.capitalize()
    )

ax.set_xlabel("Time")
ax.set_ylabel(r"$\mathbb{E}[|\mathbf{v}\cdot\Delta\mathbf{s}|]$")
ax.set_title("Mean Absolute Velocity–Score Integrand")

ax.legend(frameon=False)

fig.tight_layout()

save_figure(fig, "05_mean_absolute_integrand")


# ============================================================
# FIGURE 6: INTEGRAND STANDARD DEVIATION
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6))

for flow in FLOWS:

    times, mean, std = get_mean_std(
        flow,
        "std_integrand"
    )

    if mean is None:
        continue

    ax.plot(
        times,
        mean,
        color=COLORS[flow],
        linewidth=2.3,
        label=flow.capitalize()
    )

ax.set_xlabel("Time")
ax.set_ylabel("Integrand standard deviation")
ax.set_title("Variability of Velocity–Score Interaction")

ax.legend(frameon=False)

fig.tight_layout()

save_figure(fig, "06_integrand_variability")


# ============================================================
# FIGURE 7: CHECK COORDINATE DECOMPOSITION
# ============================================================

print("\nCOORDINATE DECOMPOSITION CHECK")
print("=" * 65)

for flow in FLOWS:

    errors = []

    for pair, data in DATA[flow].items():

        reconstructed = (
            data["mean_x_contribution"]
            + data["mean_y_contribution"]
        )

        original = data["theoretical_derivative"]

        error = np.nanmean(
            np.abs(reconstructed - original)
        )

        errors.append(error)

    if errors:
        print(
            f"{flow:18s} "
            f"mean absolute decomposition error = "
            f"{np.mean(errors):.8e}"
        )


print("\nAll plots saved in:", OUTPUT_DIR)

linear: loaded 19 pairs
trigonometric: loaded 19 pairs
spherical: loaded 19 pairs
operator: loaded 19 pairs

COORDINATE DECOMPOSITION CHECK
linear             mean absolute decomposition error = 1.53997232e-15
trigonometric      mean absolute decomposition error = 1.44646336e-15
spherical          mean absolute decomposition error = 1.46203368e-15
operator           mean absolute decomposition error = 1.81886324e-15

All plots saved in: velocity_score_plots


In [1]:

import os
import numpy as np
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt


# ============================================================
# CONFIGURATION
# ============================================================

INPUT_DIR = "velocity_score_kl_analysis"
OUTPUT_DIR = "velocity_score_plots_all_pairs"

FLOWS = [
    "linear",
    "trigonometric",
    "spherical",
    "operator"
]

N_PAIRS = 19
DPI = 600

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# PLOT STYLE
# ============================================================
plt.style.use("default")

FLOW_COLORS = {
    "linear": "#0072B2",
    "trigonometric": "#E69F00",
    "spherical": "#009E73",
    "operator": "#D55E00"
}

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "font.weight": "bold",

    "axes.labelsize": 14,
    "axes.labelweight": "bold",
    "axes.titlesize": 15,
    "axes.titleweight": "bold",

    "legend.fontsize": 11,

    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",

    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",

    "xtick.color": "black",
    "ytick.color": "black",

    "axes.linewidth": 1.8,
    "xtick.major.width": 1.8,
    "ytick.major.width": 1.8,
    "xtick.major.size": 6,
    "ytick.major.size": 6,

    "axes.grid": False,
    "lines.linewidth": 2.5,

    "savefig.dpi": 400,
    "savefig.bbox": "tight"
})


# ============================================================
# PAIR COLORS
# ============================================================

PAIR_COLORS = plt.cm.turbo(
    np.linspace(0, 1, N_PAIRS)
)



# ============================================================
# LOAD DATA
# ============================================================

DATA = {}

print("\nLoading NPZ files...")
print("=" * 70)

for flow in FLOWS:

    DATA[flow] = {}

    for pair_idx in range(1, N_PAIRS + 1):

        pair = f"pair_{pair_idx:02d}"

        file_path = os.path.join(
            INPUT_DIR,
            flow,
            pair,
            "velocity_score_kl_analysis.npz"
        )

        if not os.path.exists(file_path):
            print(f"MISSING: {file_path}")
            continue

        with np.load(file_path) as npz:

            DATA[flow][pair] = {
                key: npz[key].copy()
                for key in npz.files
            }

    print(
        f"{flow:18s}: "
        f"{len(DATA[flow])}/{N_PAIRS} pairs loaded"
    )


# ============================================================
# SAVE FUNCTION
# ============================================================

def save_figure(fig, filename):

    png_path = os.path.join(
        OUTPUT_DIR,
        filename + ".png"
    )

    pdf_path = os.path.join(
        OUTPUT_DIR,
        filename + ".pdf"
    )

    fig.savefig(
        png_path,
        dpi=DPI,
        bbox_inches="tight"
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight"
    )

    plt.close(fig)

    print("Saved:", filename)


# ============================================================
# COMMON PLOTTING FUNCTION
# ============================================================

def plot_all_pairs(
    flow,
    key,
    ylabel,
    title,
    filename,
    zero_line=True
):

    fig, ax = plt.subplots(figsize=(12, 7))

    for pair_idx in range(1, N_PAIRS + 1):

        pair = f"pair_{pair_idx:02d}"

        if pair not in DATA[flow]:
            continue

        d = DATA[flow][pair]

        if key not in d:
            continue

        ax.plot(
            d["time"],
            d[key],
            color=PAIR_COLORS[pair_idx - 1],
            linewidth=1.6,
            alpha=0.9,
            label=pair
        )

    if zero_line:

        ax.axhline(
            0,
            color="white",
            linestyle="--",
            linewidth=1.0,
            alpha=0.65
        )

    ax.set_xlabel("Time")
    ax.set_ylabel(ylabel)

    ax.set_title(
        f"{flow.capitalize()}: {title}",
        pad=12
    )

    ax.legend(
        loc="best",
        ncol=4,
        fontsize=8,
        frameon=False
    )

    fig.tight_layout()

    save_figure(fig, filename)


# ============================================================
# FIGURES 1-6: INDIVIDUAL PATHS, ALL 19 PAIRS
# ============================================================

for flow in FLOWS:

    print("\nGenerating figures for:", flow)

    # --------------------------------------------------------
    # 1. VELOCITY-SCORE CONTRIBUTION
    # --------------------------------------------------------

    plot_all_pairs(
        flow=flow,
        key="theoretical_derivative",
        ylabel=r"$\mathbb{E}_{p_t}[\mathbf{v}_t\cdot\Delta\mathbf{s}_t]$",
        title="Velocity-Score Contribution",
        filename=f"01_velocity_score_{flow}"
    )

    # --------------------------------------------------------
    # 2. PSI CONTRIBUTION
    # --------------------------------------------------------

    plot_all_pairs(
        flow=flow,
        key="mean_x_contribution",
        ylabel=r"$\mathbb{E}[v_\psi\Delta s_\psi]$",
        title="Psi Coordinate Contribution",
        filename=f"02_psi_contribution_{flow}"
    )

    # --------------------------------------------------------
    # 3. PHI CONTRIBUTION
    # --------------------------------------------------------

    plot_all_pairs(
        flow=flow,
        key="mean_y_contribution",
        ylabel=r"$\mathbb{E}[v_\phi\Delta s_\phi]$",
        title="Phi Coordinate Contribution",
        filename=f"03_phi_contribution_{flow}"
    )

    # --------------------------------------------------------
    # 4. FINITE DIFFERENCE KL DERIVATIVE
    # --------------------------------------------------------

    plot_all_pairs(
        flow=flow,
        key="finite_difference_kl",
        ylabel=r"$dD_{\mathrm{KL}}/dt$",
        title="Finite-Difference KL Derivative",
        filename=f"04_finite_difference_kl_{flow}"
    )

    # --------------------------------------------------------
    # 5. MEAN ABSOLUTE INTEGRAND
    # --------------------------------------------------------

    plot_all_pairs(
        flow=flow,
        key="mean_absolute_integrand",
        ylabel=r"$\mathbb{E}[|\mathbf{v}\cdot\Delta\mathbf{s}|]$",
        title="Mean Absolute Integrand",
        filename=f"05_absolute_integrand_{flow}",
        zero_line=False
    )

    # --------------------------------------------------------
    # 6. INTEGRAND STANDARD DEVIATION
    # --------------------------------------------------------

    plot_all_pairs(
        flow=flow,
        key="std_integrand",
        ylabel="Integrand standard deviation",
        title="Velocity-Score Integrand Variability",
        filename=f"06_integrand_std_{flow}",
        zero_line=False
    )


# ============================================================
# FIGURE 7: KL DERIVATIVE COMPARISON
# ALL PAIRS, THEORETICAL VS FINITE DIFFERENCE
# ============================================================

for flow in FLOWS:

    fig, axes = plt.subplots(
        1, 2,
        figsize=(16, 7),
        sharex=True
    )

    ax1, ax2 = axes

    for pair_idx in range(1, N_PAIRS + 1):

        pair = f"pair_{pair_idx:02d}"

        if pair not in DATA[flow]:
            continue

        d = DATA[flow][pair]

        color = PAIR_COLORS[pair_idx - 1]

        ax1.plot(
            d["time"],
            d["theoretical_derivative"],
            color=color,
            linewidth=1.4,
            alpha=0.85,
            label=pair
        )

        ax2.plot(
            d["time"],
            d["finite_difference_kl"],
            color=color,
            linewidth=1.4,
            alpha=0.85,
            label=pair
        )

    for ax in axes:

        ax.axhline(
            0,
            color="white",
            linestyle="--",
            alpha=0.6
        )

        ax.set_xlabel("Time")
        ax.legend(
            ncol=3,
            fontsize=7,
            frameon=False
        )

    ax1.set_title("Velocity-Score Derivative")
    ax1.set_ylabel(r"$dD_{\mathrm{KL}}/dt$")

    ax2.set_title("Finite-Difference KL Derivative")
    ax2.set_ylabel(r"$dD_{\mathrm{KL}}/dt$")

    fig.suptitle(
        f"{flow.capitalize()}: Theoretical vs Numerical Derivatives",
        fontsize=15
    )

    fig.tight_layout()

    save_figure(
        fig,
        f"07_derivative_comparison_{flow}"
    )


# ============================================================
# FIGURE 8: FOUR-PANEL VELOCITY-SCORE COMPARISON
# ALL 19 PAIRS IN EACH PANEL
# ============================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(16, 11),
    sharex=True,
    sharey=True
)

axes = axes.flatten()

for ax, flow in zip(axes, FLOWS):

    for pair_idx in range(1, N_PAIRS + 1):

        pair = f"pair_{pair_idx:02d}"

        if pair not in DATA[flow]:
            continue

        d = DATA[flow][pair]

        ax.plot(
            d["time"],
            d["theoretical_derivative"],
            color=PAIR_COLORS[pair_idx - 1],
            linewidth=1.25,
            alpha=0.85
        )

    ax.axhline(
        0,
        color="white",
        linestyle="--",
        alpha=0.6
    )

    ax.set_title(flow.capitalize())
    ax.set_xlabel("Time")
    ax.set_ylabel(r"$\mathbb{E}[\mathbf{v}\cdot\Delta\mathbf{s}]$")

fig.suptitle(
    "Velocity-Score Contributions Across 19 Molecular Pairs",
    fontsize=16
)

fig.tight_layout()

save_figure(
    fig,
    "08_four_path_all_pairs_velocity_score"
)


# ============================================================
# FIGURE 9: FOUR-PANEL FINITE-DIFFERENCE KL COMPARISON
# ALL 19 PAIRS IN EACH PANEL
# ============================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(16, 11),
    sharex=True,
    sharey=True
)

axes = axes.flatten()

for ax, flow in zip(axes, FLOWS):

    for pair_idx in range(1, N_PAIRS + 1):

        pair = f"pair_{pair_idx:02d}"

        if pair not in DATA[flow]:
            continue

        d = DATA[flow][pair]

        ax.plot(
            d["time"],
            d["finite_difference_kl"],
            color=PAIR_COLORS[pair_idx - 1],
            linewidth=1.25,
            alpha=0.85
        )

    ax.axhline(
        0,
        color="white",
        linestyle="--",
        alpha=0.6
    )

    ax.set_title(flow.capitalize())
    ax.set_xlabel("Time")
    ax.set_ylabel(r"$dD_{\mathrm{KL}}/dt$")

fig.suptitle(
    "Finite-Difference KL Derivatives Across 19 Molecular Pairs",
    fontsize=16
)

fig.tight_layout()

save_figure(
    fig,
    "09_four_path_all_pairs_finite_difference"
)


# ============================================================
# FIGURE 10: FOUR-PANEL PSI CONTRIBUTION
# ============================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(16, 11),
    sharex=True,
    sharey=True
)

axes = axes.flatten()

for ax, flow in zip(axes, FLOWS):

    for pair_idx in range(1, N_PAIRS + 1):

        pair = f"pair_{pair_idx:02d}"

        if pair not in DATA[flow]:
            continue

        d = DATA[flow][pair]

        ax.plot(
            d["time"],
            d["mean_x_contribution"],
            color=PAIR_COLORS[pair_idx - 1],
            linewidth=1.2,
            alpha=0.85
        )

    ax.axhline(0, color="white", linestyle="--", alpha=0.6)

    ax.set_title(flow.capitalize())
    ax.set_xlabel("Time")
    ax.set_ylabel(r"$A_\psi(t)$")

fig.suptitle(
    "Psi Contributions Across All Molecular Pairs",
    fontsize=16
)

fig.tight_layout()

save_figure(
    fig,
    "10_four_path_all_pairs_psi"
)


# ============================================================
# FIGURE 11: FOUR-PANEL PHI CONTRIBUTION
# ============================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(16, 11),
    sharex=True,
    sharey=True
)

axes = axes.flatten()

for ax, flow in zip(axes, FLOWS):

    for pair_idx in range(1, N_PAIRS + 1):

        pair = f"pair_{pair_idx:02d}"

        if pair not in DATA[flow]:
            continue

        d = DATA[flow][pair]

        ax.plot(
            d["time"],
            d["mean_y_contribution"],
            color=PAIR_COLORS[pair_idx - 1],
            linewidth=1.2,
            alpha=0.85
        )

    ax.axhline(0, color="white", linestyle="--", alpha=0.6)

    ax.set_title(flow.capitalize())
    ax.set_xlabel("Time")
    ax.set_ylabel(r"$A_\phi(t)$")

fig.suptitle(
    "Phi Contributions Across All Molecular Pairs",
    fontsize=16
)

fig.tight_layout()

save_figure(
    fig,
    "11_four_path_all_pairs_phi"
)


# ============================================================
# DECOMPOSITION VALIDATION
# ============================================================

print("\n" + "=" * 75)
print("VELOCITY-SCORE DECOMPOSITION VALIDATION")
print("=" * 75)

for flow in FLOWS:

    errors = []

    for pair, d in DATA[flow].items():

        reconstructed = (
            d["mean_x_contribution"]
            + d["mean_y_contribution"]
        )

        original = d["theoretical_derivative"]

        error = np.nanmean(
            np.abs(reconstructed - original)
        )

        errors.append(error)

    if errors:

        print(
            f"{flow:18s} | "
            f"Mean absolute decomposition error: "
            f"{np.mean(errors):.8e}"
        )


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("ANALYSIS COMPLETE")
print("=" * 75)

print("Output directory:", OUTPUT_DIR)

print("Figures generated:")
print("01-06: Individual probability paths, all pairs")
print("07: Theoretical vs finite-difference derivatives")
print("08: Four-path velocity-score comparison")
print("09: Four-path finite-difference comparison")
print("10: Four-path psi contributions")
print("11: Four-path phi contributions")

print("\nAll figures saved as PNG and PDF at 600 DPI.")



Loading NPZ files...
linear            : 19/19 pairs loaded
trigonometric     : 19/19 pairs loaded
spherical         : 19/19 pairs loaded


Font 'nonuniit' does not have a glyph for 'E' [U+e156], substituting with a dummy symbol.
Font 'nonuniit' does not have a glyph for 'E' [U+e156], substituting with a dummy symbol.


operator          : 19/19 pairs loaded

Generating figures for: linear


Font 'nonuniit' does not have a glyph for 'E' [U+e156], substituting with a dummy symbol.
Font 'nonuniit' does not have a glyph for 'E' [U+e156], substituting with a dummy symbol.
Font 'nonuniit' does not have a glyph for 'E' [U+e156], substituting with a dummy symbol.
Font 'nonuniit' does not have a glyph for 'E' [U+e156], substituting with a dummy symbol.
Font 'nonuniit' does not have a glyph for 'E' [U+e156], substituting with a dummy symbol.
Font 'nonuniit' does not have a glyph for 'E' [U+e156], substituting with a dummy symbol.


Saved: 01_velocity_score_linear


Font 'nonuniit' does not have a glyph for 'E' [U+e156], substituting with a dummy symbol.
Font 'nonuniit' does not have a glyph for 'E' [U+e156], substituting with a dummy symbol.
Font 'nonuniit' does not have a glyph for 'E' [U+e156], substituting with a dummy symbol.
Font 'nonuniit' does not have a glyph for 'E' [U+e156], substituting with a dummy symbol.
Font 'nonuniit' does not have a glyph for 'E' [U+e156], substituting with a dummy symbol.


... (43 outputs left out)

Font 'nonuniit' does not have a glyph for 'E' [U+e156], substituting with a dummy symbol.


Saved: 07_derivative_comparison_operator


Font 'nonuniit' does not have a glyph for 'E' [U+e156], substituting with a dummy symbol.
Font 'nonuniit' does not have a glyph for 'E' [U+e156], substituting with a dummy symbol.
Font 'nonuniit' does not have a glyph for 'E' [U+e156], substituting with a dummy symbol.
Font 'nonuniit' does not have a glyph for 'E' [U+e156], substituting with a dummy symbol.


Saved: 08_four_path_all_pairs_velocity_score
Saved: 09_four_path_all_pairs_finite_difference
Saved: 10_four_path_all_pairs_psi
Saved: 11_four_path_all_pairs_phi

VELOCITY-SCORE DECOMPOSITION VALIDATION
linear             | Mean absolute decomposition error: 1.53997232e-15
trigonometric      | Mean absolute decomposition error: 1.44646336e-15
spherical          | Mean absolute decomposition error: 1.46203368e-15
operator           | Mean absolute decomposition error: 1.81886324e-15

ANALYSIS COMPLETE
Output directory: velocity_score_plots_all_pairs
Figures generated:
01-06: Individual probability paths, all pairs
07: Theoretical vs finite-difference derivatives
08: Four-path velocity-score comparison
09: Four-path finite-difference comparison
10: Four-path psi contributions
11: Four-path phi contributions

All figures saved as PNG and PDF at 600 DPI.


# KL reconstruction analysis 

In [6]:
import os
import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable


# ============================================================
# CONFIGURATION
# ============================================================

INPUT_DIR = "velocity_score_kl_analysis"
OUTPUT_DIR = "kl_all_reconstructions"

FLOWS = [
    "linear",
    "trigonometric",
    "spherical",
    "operator"
]

N_PAIRS = 19
DPI = 600

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# PUBLICATION STYLE
# ============================================================
plt.style.use("default")

COLORS = {
    "linear": "#0072B2",
    "trigonometric": "#E69F00",
    "spherical": "#009E73",
    "operator": "#D55E00"
}

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "font.weight": "bold",

    "axes.labelsize": 14,
    "axes.labelweight": "bold",
    "axes.titlesize": 15,
    "axes.titleweight": "bold",

    "legend.fontsize": 11,

    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",

    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",

    "xtick.color": "black",
    "ytick.color": "black",

    "axes.linewidth": 1.8,
    "xtick.major.width": 1.8,
    "ytick.major.width": 1.8,
    "xtick.major.size": 6,
    "ytick.major.size": 6,

    "axes.grid": False,
    "lines.linewidth": 2.5,

    "savefig.dpi": 400,
    "savefig.bbox": "tight"
})
# ============================================================
# RECONSTRUCTION METHODS
# ============================================================

METHODS = {

    "theoretical_derivative": {
        "label": r"$\int \langle v\cdot\Delta s\rangle dt$",
        "linestyle": "--",
        "marker": "o",
        "linewidth": 2.0
    },

    "finite_difference_kl": {
        "label": "Finite-difference reconstruction",
        "linestyle": ":",
        "marker": "^",
        "linewidth": 2.0
    },

    "temporal_flux_derivative": {
        "label": "Temporal-flux reconstruction",
        "linestyle": "-.",
        "marker": "s",
        "linewidth": 2.0
    }

}


# ============================================================
# KL RECONSTRUCTION
# ============================================================

def reconstruct_kl(time, derivative, initial_kl):

    """
    KL(t) = KL(0) + integral_0^t derivative(t) dt

    Cumulative trapezoidal integration.
    """

    reconstructed = np.zeros_like(
        derivative,
        dtype=float
    )

    reconstructed[0] = initial_kl

    dt = np.diff(time)

    increments = (
        0.5 *
        (derivative[:-1] + derivative[1:]) *
        dt
    )

    reconstructed[1:] = (
        initial_kl + np.cumsum(increments)
    )

    return reconstructed


# ============================================================
# PAIR COLOR MAP
# ============================================================

cmap = plt.cm.turbo

norm = Normalize(
    vmin=1,
    vmax=N_PAIRS
)

scalar_map = ScalarMappable(
    norm=norm,
    cmap=cmap
)


# ============================================================
# PROCESS EACH FLOW
# ============================================================

for flow in FLOWS:

    print(f"\nProcessing: {flow}")

    fig, ax = plt.subplots(
        figsize=(15, 9)
    )

    # --------------------------------------------------------
    # LOOP OVER ALL 19 PAIRS
    # --------------------------------------------------------

    for pair in range(1, N_PAIRS + 1):

        npz_path = os.path.join(
            INPUT_DIR,
            flow,
            f"pair_{pair:02d}",
            "velocity_score_kl_analysis.npz"
        )

        if not os.path.exists(npz_path):

            print(f"Missing: {npz_path}")
            continue

        data = np.load(npz_path)

        time = data["time"]

        original_kl = data["kl"]

        color = cmap(norm(pair))

        # ----------------------------------------------------
        # ORIGINAL KL
        # ----------------------------------------------------

        ax.plot(
            time,
            original_kl,

            color=color,

            linestyle="-",

            linewidth=2.8,

            alpha=0.95,

            zorder=5
        )

        # ----------------------------------------------------
        # ALL THREE RECONSTRUCTIONS
        # ----------------------------------------------------

        for method_key, method_style in METHODS.items():

            if method_key not in data:
                print(
                    f"Missing {method_key} "
                    f"in pair {pair:02d}"
                )
                continue

            derivative = data[method_key]

            reconstructed_kl = reconstruct_kl(
                time,
                derivative,
                original_kl[0]
            )

            ax.plot(

                time,

                reconstructed_kl,

                color=color,

                linestyle=method_style["linestyle"],

                linewidth=method_style["linewidth"],

                marker=method_style["marker"],

                markersize=3.5,

                markevery=20,

                markeredgewidth=0.8,

                alpha=0.85,

                zorder=3

            )

        print(f"  Pair {pair:02d} plotted")

        data.close()

    # ========================================================
    # AXIS FORMATTING
    # ========================================================

    ax.set_xlabel(
        r"Time ($t$)",
        fontsize=21,
        fontweight="bold"
    )

    ax.set_ylabel(
        r"$D_{\mathrm{KL}}(p_t\|p_{\mathrm{ref}})$",
        fontsize=21,
        fontweight="bold"
    )

    ax.set_title(
        f"{flow.capitalize()} Flow",
        fontsize=25,
        fontweight="bold",
        pad=18
    )

    ax.set_xlim(0, 1)

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=17,
        width=2.5,
        length=9,
        direction="out"
    )

    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("bold")

    for spine in ax.spines.values():
        spine.set_linewidth(2.5)
        spine.set_color("black")

    ax.grid(False)

    # ========================================================
    # PAIR COLOR BAR
    # ========================================================

    cbar = fig.colorbar(
        scalar_map,
        ax=ax,
        pad=0.025,
        fraction=0.035
    )

    cbar.set_label(
        "ψ–φ Pair",
        fontsize=18,
        fontweight="bold",
        labelpad=12
    )

    cbar.set_ticks(
        np.arange(1, N_PAIRS + 1)
    )

    cbar.set_ticklabels(
        [f"{i:02d}" for i in range(1, N_PAIRS + 1)]
    )

    cbar.ax.tick_params(
        labelsize=12,
        width=2,
        length=5
    )

    for label in cbar.ax.get_yticklabels():
        label.set_fontweight("bold")

    cbar.outline.set_linewidth(2)

    # ========================================================
    # CUSTOM LINE-STYLE LEGEND
    # ========================================================

    from matplotlib.lines import Line2D

    legend_handles = [

        Line2D(
            [0], [0],
            color="black",
            linewidth=2.8,
            linestyle="-",
            label="Original KL"
        )

    ]

    for method_style in METHODS.values():

        legend_handles.append(

            Line2D(
                [0], [0],

                color="black",

                linewidth=2.2,

                linestyle=method_style["linestyle"],

                marker=method_style["marker"],

                markersize=6,

                label=method_style["label"]
            )

        )

    legend = ax.legend(
        handles=legend_handles,
        loc="best",
        fontsize=12,
        frameon=False,
        ncol=2,
        handlelength=3,
        columnspacing=1.5
    )

    for label in legend.get_texts():
        label.set_fontweight("bold")

    # ========================================================
    # SAVE FIGURE
    # ========================================================

    plt.tight_layout()

    png_path = os.path.join(
        OUTPUT_DIR,
        f"{flow}_all_pairs_all_reconstructions.png"
    )

    pdf_path = os.path.join(
        OUTPUT_DIR,
        f"{flow}_all_pairs_all_reconstructions.pdf"
    )

    fig.savefig(
        png_path,
        dpi=DPI,
        bbox_inches="tight"
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight"
    )

    plt.close(fig)

    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")


# ============================================================
# COMPLETION
# ============================================================

print("\n" + "=" * 70)
print("ALL RECONSTRUCTION FIGURES GENERATED")
print("=" * 70)

print(f"Output directory: {OUTPUT_DIR}")
print("Total figures: 4")
print("Pairs per figure: 19")
print("Curves per pair: 4")
print("Total plotted trajectories: 304")


Processing: linear
  Pair 01 plotted
  Pair 02 plotted
  Pair 03 plotted
  Pair 04 plotted
  Pair 05 plotted
  Pair 06 plotted
  Pair 07 plotted
  Pair 08 plotted
  Pair 09 plotted
  Pair 10 plotted
  Pair 11 plotted
  Pair 12 plotted
  Pair 13 plotted
  Pair 14 plotted
  Pair 15 plotted
  Pair 16 plotted
  Pair 17 plotted
  Pair 18 plotted
  Pair 19 plotted
Saved: kl_all_reconstructions/linear_all_pairs_all_reconstructions.png
Saved: kl_all_reconstructions/linear_all_pairs_all_reconstructions.pdf

Processing: trigonometric
  Pair 01 plotted
  Pair 02 plotted
  Pair 03 plotted
  Pair 04 plotted
  Pair 05 plotted
  Pair 06 plotted
  Pair 07 plotted
  Pair 08 plotted
  Pair 09 plotted
  Pair 10 plotted
  Pair 11 plotted
  Pair 12 plotted
  Pair 13 plotted
  Pair 14 plotted
  Pair 15 plotted
  Pair 16 plotted
  Pair 17 plotted
  Pair 18 plotted
  Pair 19 plotted
Saved: kl_all_reconstructions/trigonometric_all_pairs_all_reconstructions.png
Saved: kl_all_reconstructions/trigonometric_all_p

In [13]:
# vELOCITY ANALYSIS

In [7]:
import os
import glob
import numpy as np

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm


# ============================================================
# CONFIGURATION
# ============================================================

INPUT_DIR = "velocity_score_kl_analysis"
OUTPUT_DIR = "kl_explanatory_plots"

FLOWS = [
    "linear",
    "trigonometric",
    "spherical",
    "operator"
]

SNAPSHOT_TIMES = [0.0, 0.25, 0.50, 0.75, 1.0]

DPI = 400

# Numerical stability for logarithms
EPS = 1e-12

plt.style.use("default")

COLORS = {
    "linear": "#0072B2",
    "trigonometric": "#E69F00",
    "spherical": "#009E73",
    "operator": "#D55E00"
}

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "font.weight": "bold",

    "axes.labelsize": 14,
    "axes.labelweight": "bold",
    "axes.titlesize": 15,
    "axes.titleweight": "bold",

    "legend.fontsize": 11,

    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",

    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",

    "xtick.color": "black",
    "ytick.color": "black",

    "axes.linewidth": 1.8,
    "xtick.major.width": 1.8,
    "ytick.major.width": 1.8,
    "xtick.major.size": 6,
    "ytick.major.size": 6,

    "axes.grid": False,
    "lines.linewidth": 2.5,

    "savefig.dpi": 400,
    "savefig.bbox": "tight"
})

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def style_axis(ax):

    for spine in ax.spines.values():
        spine.set_linewidth(2.0)

    ax.tick_params(
        axis="both",
        which="major",
        width=2,
        length=6,
        labelsize=11
    )

    ax.grid(False)


def integrate_2d(values, x_axis, y_axis):
    """
    Integrate a 2D grid whose shape is (len(x), len(y)).
    """

    integral_y = np.trapz(
        values,
        x=y_axis,
        axis=1
    )

    integral = np.trapz(
        integral_y,
        x=x_axis,
        axis=0
    )

    return integral


def get_snapshot_indices(time):

    indices = []

    for target in SNAPSHOT_TIMES:

        idx = np.argmin(np.abs(time - target))

        indices.append(idx)

    return indices


def save_figure(fig, path):

    fig.savefig(
        path,
        dpi=DPI,
        bbox_inches="tight"
    )

    plt.close(fig)


# ============================================================
# 1. KL DIVERGENCE VS TIME
# ============================================================

def plot_kl(time, kl, output_path, flow, pair):

    fig, ax = plt.subplots(figsize=(8, 6))

    ax.plot(
        time,
        kl,
        linewidth=3,
        color="black"
    )

    ax.scatter(
        time[::10],
        kl[::10],
        s=35,
        marker="o",
        color="black",
        zorder=3
    )

    ax.set_xlabel(
        "Time",
        fontsize=15,
        fontweight="bold"
    )

    ax.set_ylabel(
        r"$D_{\mathrm{KL}}(p_t\Vert p_{\mathrm{ref}})$",
        fontsize=15,
        fontweight="bold"
    )

    ax.set_title(
        f"{flow.capitalize()} | Pair {pair:02d}",
        fontsize=16
    )

    style_axis(ax)

    save_figure(fig, output_path)


# ============================================================
# 2. JOINT DENSITY EVOLUTION
# ============================================================

def plot_density_evolution(
    x_axis,
    y_axis,
    p_ref,
    p_t,
    time,
    indices,
    output_path,
    flow,
    pair
):

    fig, axes = plt.subplots(
        2,
        3,
        figsize=(17, 10)
    )

    axes = axes.flatten()

    # Reference + generated densities
    all_densities = [p_ref]

    for idx in indices:
        all_densities.append(p_t[idx])

    vmax = max(np.max(p) for p in all_densities)

    levels = np.linspace(0, vmax, 25)

    # Reference distribution
    im = axes[0].contourf(
        x_axis,
        y_axis,
        p_ref.T,
        levels=levels,
        cmap="viridis"
    )

    axes[0].set_title(
        "Reference distribution",
        fontsize=14
    )

    axes[0].set_xlabel(r"$\psi$")
    axes[0].set_ylabel(r"$\phi$")

    style_axis(axes[0])

    # Generated distributions
    for j, idx in enumerate(indices):

        ax = axes[j + 1]

        im = ax.contourf(
            x_axis,
            y_axis,
            p_t[idx].T,
            levels=levels,
            cmap="viridis"
        )

        ax.set_title(
            f"Generated: t = {time[idx]:.2f}",
            fontsize=14
        )

        ax.set_xlabel(r"$\psi$")
        ax.set_ylabel(r"$\phi$")

        style_axis(ax)

    # Shared colorbar
    fig.colorbar(
        im,
        ax=axes.tolist(),
        shrink=0.85,
        label="Probability density"
    )

    fig.suptitle(
        f"Joint Density Evolution | {flow.capitalize()} | Pair {pair:02d}",
        fontsize=18,
        fontweight="bold"
    )

    save_figure(fig, output_path)


# ============================================================
# 3. LOG-DENSITY RATIO
# ============================================================

def plot_log_density_ratio(
    x_axis,
    y_axis,
    p_ref,
    p_t,
    time,
    indices,
    output_path,
    flow,
    pair
):

    log_ratios = []

    for idx in indices:

        pt = np.maximum(p_t[idx], EPS)
        pref = np.maximum(p_ref, EPS)

        log_ratio = np.log(pt / pref)

        log_ratios.append(log_ratio)

    # Use common symmetric color limits
    all_values = np.concatenate([
        z.ravel() for z in log_ratios
    ])

    vmax = np.percentile(np.abs(all_values), 99)

    vmax = max(vmax, 1e-6)

    norm = TwoSlopeNorm(
        vmin=-vmax,
        vcenter=0,
        vmax=vmax
    )

    fig, axes = plt.subplots(
        2,
        3,
        figsize=(17, 10)
    )

    axes = axes.flatten()

    for j, idx in enumerate(indices):

        ax = axes[j]

        im = ax.contourf(
            x_axis,
            y_axis,
            log_ratios[j].T,
            levels=30,
            cmap="RdBu_r",
            norm=norm
        )

        ax.set_title(
            f"t = {time[idx]:.2f}",
            fontsize=14
        )

        ax.set_xlabel(r"$\psi$")
        ax.set_ylabel(r"$\phi$")

        style_axis(ax)

    # Disable unused panel
    axes[-1].axis("off")

    fig.colorbar(
        im,
        ax=axes[:-1].tolist(),
        shrink=0.85,
        label=r"$\log(p_t/p_{\mathrm{ref}})$"
    )

    fig.suptitle(
        f"Log-Density Ratio | {flow.capitalize()} | Pair {pair:02d}",
        fontsize=18,
        fontweight="bold"
    )

    save_figure(fig, output_path)


# ============================================================
# 4. SPATIAL KL CONTRIBUTION
# ============================================================

def plot_spatial_kl(
    x_axis,
    y_axis,
    p_ref,
    p_t,
    time,
    indices,
    output_path,
    flow,
    pair
):

    contributions = []

    for idx in indices:

        pt = np.maximum(p_t[idx], EPS)
        pref = np.maximum(p_ref, EPS)

        # Local KL integrand
        contribution = pt * np.log(pt / pref)

        contributions.append(contribution)

    # Common symmetric color scale
    all_values = np.concatenate([
        z.ravel() for z in contributions
    ])

    vmax = np.percentile(
        np.abs(all_values),
        99
    )

    vmax = max(vmax, 1e-12)

    norm = TwoSlopeNorm(
        vmin=-vmax,
        vcenter=0,
        vmax=vmax
    )

    fig, axes = plt.subplots(
        2,
        3,
        figsize=(17, 10)
    )

    axes = axes.flatten()

    for j, idx in enumerate(indices):

        ax = axes[j]

        im = ax.contourf(
            x_axis,
            y_axis,
            contributions[j].T,
            levels=30,
            cmap="RdBu_r",
            norm=norm
        )

        # Calculate integrated KL from spatial contribution
        kl_integrated = integrate_2d(
            contributions[j],
            x_axis,
            y_axis
        )

        ax.set_title(
            f"t = {time[idx]:.2f} | KL = {kl_integrated:.4f}",
            fontsize=13
        )

        ax.set_xlabel(r"$\psi$")
        ax.set_ylabel(r"$\phi$")

        style_axis(ax)

    axes[-1].axis("off")

    fig.colorbar(
        im,
        ax=axes[:-1].tolist(),
        shrink=0.85,
        label=r"$p_t\log(p_t/p_{\mathrm{ref}})$"
    )

    fig.suptitle(
        f"Spatial KL Contribution | {flow.capitalize()} | Pair {pair:02d}",
        fontsize=18,
        fontweight="bold"
    )

    save_figure(fig, output_path)


# ============================================================
# MAIN
# ============================================================

def main():

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )

    total_pairs = 0

    for flow in FLOWS:

        print("\n" + "=" * 65)
        print(f"FLOW: {flow}")
        print("=" * 65)

        pattern = os.path.join(
            INPUT_DIR,
            flow,
            "pair_*",
            "velocity_score_kl_analysis.npz"
        )

        files = sorted(glob.glob(pattern))

        if not files:

            print(f"No files found for {flow}")
            continue

        for file_path in files:

            pair_name = os.path.basename(
                os.path.dirname(file_path)
            )

            pair = int(pair_name.split("_")[-1])

            print(f"Processing {flow} | Pair {pair:02d}")

            data = np.load(file_path)

            # ------------------------------------------------
            # LOAD DATA
            # ------------------------------------------------

            time = data["time"]

            kl = data["kl"]

            p_ref = data["p_ref"]

            p_t = data["p_t"]

            x_axis = data["x_axis"]

            y_axis = data["y_axis"]

            # ------------------------------------------------
            # VALIDATE SHAPES
            # ------------------------------------------------

            if p_t.shape[0] != len(time):

                print(
                    f"Skipping {flow} pair {pair}: "
                    "time and density dimensions do not match."
                )

                continue

            if p_ref.shape != p_t.shape[1:]:

                print(
                    f"Skipping {flow} pair {pair}: "
                    "reference and generated density shapes differ."
                )

                continue

            # ------------------------------------------------
            # OUTPUT DIRECTORY
            # ------------------------------------------------

            pair_output = os.path.join(
                OUTPUT_DIR,
                flow,
                pair_name
            )

            os.makedirs(
                pair_output,
                exist_ok=True
            )

            indices = get_snapshot_indices(time)

            # ------------------------------------------------
            # FIGURE 1: KL VS TIME
            # ------------------------------------------------

            plot_kl(
                time,
                kl,
                os.path.join(
                    pair_output,
                    "01_kl_vs_time.png"
                ),
                flow,
                pair
            )

            # ------------------------------------------------
            # FIGURE 2: JOINT DENSITY EVOLUTION
            # ------------------------------------------------

            plot_density_evolution(
                x_axis,
                y_axis,
                p_ref,
                p_t,
                time,
                indices,
                os.path.join(
                    pair_output,
                    "02_joint_density_evolution.png"
                ),
                flow,
                pair
            )

            # ------------------------------------------------
            # FIGURE 3: LOG DENSITY RATIO
            # ------------------------------------------------

            plot_log_density_ratio(
                x_axis,
                y_axis,
                p_ref,
                p_t,
                time,
                indices,
                os.path.join(
                    pair_output,
                    "03_log_density_ratio.png"
                ),
                flow,
                pair
            )

            # ------------------------------------------------
            # FIGURE 4: SPATIAL KL CONTRIBUTION
            # ------------------------------------------------

            plot_spatial_kl(
                x_axis,
                y_axis,
                p_ref,
                p_t,
                time,
                indices,
                os.path.join(
                    pair_output,
                    "04_spatial_kl_contribution.png"
                ),
                flow,
                pair
            )

            total_pairs += 1

            data.close()

    print("\n" + "=" * 65)
    print("PLOTTING COMPLETED")
    print("=" * 65)

    print(f"Total pairs processed: {total_pairs}")

    print(f"Output directory: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()


FLOW: linear
Processing linear | Pair 01
Processing linear | Pair 02
Processing linear | Pair 03
Processing linear | Pair 04
Processing linear | Pair 05
Processing linear | Pair 06
Processing linear | Pair 07
Processing linear | Pair 08
Processing linear | Pair 09
Processing linear | Pair 10
Processing linear | Pair 11
Processing linear | Pair 12
Processing linear | Pair 13
Processing linear | Pair 14
Processing linear | Pair 15
Processing linear | Pair 16
Processing linear | Pair 17
Processing linear | Pair 18
Processing linear | Pair 19

FLOW: trigonometric
Processing trigonometric | Pair 01
Processing trigonometric | Pair 02
Processing trigonometric | Pair 03
Processing trigonometric | Pair 04
Processing trigonometric | Pair 05
Processing trigonometric | Pair 06
Processing trigonometric | Pair 07
Processing trigonometric | Pair 08
Processing trigonometric | Pair 09
Processing trigonometric | Pair 10
Processing trigonometric | Pair 11
Processing trigonometric | Pair 12
Processing tr

# Redistribution

In [20]:
import os
import glob
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ============================================================
# CONFIGURATION
# ============================================================

INPUT_DIR = "velocity_score_kl_analysis"
OUTPUT_DIR = "redistribution_entropy_publication"

FLOWS = [
    "linear",
    "trigonometric",
    "spherical",
    "operator"
]

EPS = 1e-12

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# PUBLICATION STYLE
# ============================================================

plt.rcParams.update({

    "font.family": "serif",

    "font.size": 16,
    "font.weight": "bold",

    "axes.labelsize": 19,
    "axes.labelweight": "bold",

    "axes.titlesize": 20,
    "axes.titleweight": "bold",

    "axes.linewidth": 2.5,

    "xtick.labelsize": 15,
    "ytick.labelsize": 15,

    "xtick.major.width": 2.5,
    "ytick.major.width": 2.5,

    "xtick.major.size": 7,
    "ytick.major.size": 7,

    "legend.fontsize": 10,

    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",

    "mathtext.fontset": "stix",

    "savefig.dpi": 400
})


# ============================================================
# PAIR STYLES
# ============================================================

COLORS = plt.cm.tab20(
    np.linspace(0, 1, 20)
)

LINESTYLES = [
    "-",
    "--",
    "-.",
    ":"
]

MARKERS = [
    None,
    "o",
    "s",
    "^",
    "D",
    "v",
    "P",
    "X",
    "*"
]


def pair_style(index):

    color = COLORS[index % len(COLORS)]

    linestyle = LINESTYLES[
        (index // len(COLORS)) % len(LINESTYLES)
    ]

    marker = MARKERS[
        (index // 4) % len(MARKERS)
    ]

    return color, linestyle, marker


# ============================================================
# NUMERICAL INTEGRATION
# ============================================================

def integrate_2d(array, x_axis, y_axis):

    return np.trapz(
        np.trapz(array, y_axis, axis=1),
        x_axis,
        axis=0
    )


def normalize_density(density, x_axis, y_axis):

    density = np.maximum(density, 0.0)

    norm = integrate_2d(
        density,
        x_axis,
        y_axis
    )

    if norm <= EPS:
        return density

    return density / norm


# ============================================================
# METRIC 4: DENSITY REDISTRIBUTION RATE
# ============================================================

def calculate_redistribution_rate(
    p_t,
    time,
    x_axis,
    y_axis
):

    dp_dt = np.gradient(
        p_t,
        time,
        axis=0,
        edge_order=2
    )

    redistribution = np.zeros(len(time))

    for i in range(len(time)):

        redistribution[i] = integrate_2d(
            np.abs(dp_dt[i]),
            x_axis,
            y_axis
        )

    return redistribution


# ============================================================
# METRIC 5: KL DISSIPATION AND RELATIVE DECAY
# ============================================================

def calculate_kl_rates(kl, time):

    dkl_dt = np.gradient(
        kl,
        time,
        edge_order=2
    )

    kl_dissipation = -dkl_dt

    relative_decay = np.full(
        len(kl),
        np.nan
    )

    valid = kl > EPS

    relative_decay[valid] = (
        -dkl_dt[valid] / kl[valid]
    )

    return kl_dissipation, relative_decay


# ============================================================
# METRIC 6: DIFFERENTIAL ENTROPY
# ============================================================

def calculate_entropy(
    p_t,
    p_ref,
    time,
    x_axis,
    y_axis
):

    entropy = np.zeros(len(time))

    for i in range(len(time)):

        p = normalize_density(
            p_t[i],
            x_axis,
            y_axis
        )

        entropy[i] = -integrate_2d(
            p * np.log(np.maximum(p, EPS)),
            x_axis,
            y_axis
        )

    p_ref = normalize_density(
        p_ref,
        x_axis,
        y_axis
    )

    reference_entropy = -integrate_2d(
        p_ref * np.log(np.maximum(p_ref, EPS)),
        x_axis,
        y_axis
    )

    delta_entropy = entropy - reference_entropy

    return entropy, reference_entropy, delta_entropy


# ============================================================
# LOAD ALL PAIRS FOR ONE FLOW
# ============================================================

def load_flow_data(flow):

    flow_dir = os.path.join(
        INPUT_DIR,
        flow
    )

    files = sorted(
        glob.glob(
            os.path.join(
                flow_dir,
                "pair_*",
                "velocity_score_kl_analysis.npz"
            )
        )
    )

    results = {}

    for npz_path in files:

        pair_name = os.path.basename(
            os.path.dirname(npz_path)
        )

        data = np.load(npz_path)

        time = data["time"]
        p_t = data["p_t"]
        p_ref = data["p_ref"]

        x_axis = data["x_axis"]
        y_axis = data["y_axis"]

        kl = data["kl"]

        # Metric 4
        redistribution = calculate_redistribution_rate(
            p_t,
            time,
            x_axis,
            y_axis
        )

        # Metric 5
        kl_dissipation, relative_decay = calculate_kl_rates(
            kl,
            time
        )

        # Metric 6
        entropy, reference_entropy, delta_entropy = calculate_entropy(
            p_t,
            p_ref,
            time,
            x_axis,
            y_axis
        )

        results[pair_name] = {

            "time": time,

            "kl": kl,

            "redistribution": redistribution,

            "kl_dissipation": kl_dissipation,

            "relative_decay": relative_decay,

            "entropy": entropy,

            "reference_entropy": reference_entropy,

            "delta_entropy": delta_entropy
        }

        # Save numerical results
        save_dir = os.path.join(
            OUTPUT_DIR,
            flow,
            "data"
        )

        os.makedirs(save_dir, exist_ok=True)

        np.savez_compressed(
            os.path.join(
                save_dir,
                f"{pair_name}_metrics.npz"
            ),

            time=time,
            kl=kl,
            redistribution_rate=redistribution,
            kl_dissipation=kl_dissipation,
            relative_kl_decay_rate=relative_decay,
            entropy=entropy,
            reference_entropy=reference_entropy,
            delta_entropy=delta_entropy
        )

    return results


# ============================================================
# COMMON AXIS FORMATTING
# ============================================================

def format_axis(ax):

    ax.tick_params(
        axis="both",
        which="major",
        width=2.5,
        length=7,
        labelsize=15
    )

    for label in ax.get_xticklabels():
        label.set_fontweight("bold")

    for label in ax.get_yticklabels():
        label.set_fontweight("bold")

    for spine in ax.spines.values():
        spine.set_linewidth(2.5)

    ax.grid(False)


# ============================================================
# LEGEND
# ============================================================

def add_pair_legend(ax, pair_names):

    handles, labels = ax.get_legend_handles_labels()

    ax.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.19),
        ncol=5,
        frameon=False,
        fontsize=10,
        handlelength=2.5,
        columnspacing=1.4
    )


# ============================================================
# FIGURE 1: DENSITY REDISTRIBUTION
# ============================================================

def plot_redistribution(results, flow, output_dir):

    fig, ax = plt.subplots(
        figsize=(13, 8)
    )

    pair_names = sorted(results.keys())

    for index, pair_name in enumerate(pair_names):

        d = results[pair_name]

        color, linestyle, marker = pair_style(index)

        ax.plot(
            d["time"],
            d["redistribution"],
            color=color,
            linestyle=linestyle,
            linewidth=2.5,
            marker=marker,
            markevery=25,
            markersize=5,
            label=pair_name.replace("_", " ")
        )

    ax.set_xlabel(
        "Time",
        fontweight="bold"
    )

    ax.set_ylabel(
        r"$R(t)=\int |\partial_t p_t|\,d\psi\,d\phi$",
        fontweight="bold"
    )

    ax.set_title(
        f"{flow.capitalize()}: Density Redistribution Rate",
        fontweight="bold",
        pad=15
    )

    format_axis(ax)

    add_pair_legend(ax, pair_names)

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            output_dir,
            "01_density_redistribution.png"
        ),
        dpi=400,
        bbox_inches="tight"
    )

    plt.close(fig)


# ============================================================
# FIGURE 2: KL DISSIPATION
# ============================================================

def plot_kl_dissipation(results, flow, output_dir):

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(13, 12),
        sharex=True
    )

    pair_names = sorted(results.keys())

    for index, pair_name in enumerate(pair_names):

        d = results[pair_name]

        color, linestyle, marker = pair_style(index)

        # Absolute KL dissipation
        axes[0].plot(
            d["time"],
            d["kl_dissipation"],
            color=color,
            linestyle=linestyle,
            linewidth=2.5,
            marker=marker,
            markevery=25,
            markersize=5,
            label=pair_name.replace("_", " ")
        )

        # Relative KL decay
        axes[1].plot(
            d["time"],
            d["relative_decay"],
            color=color,
            linestyle=linestyle,
            linewidth=2.5,
            marker=marker,
            markevery=25,
            markersize=5,
            label=pair_name.replace("_", " ")
        )

    axes[0].set_ylabel(
        r"$Q(t)=-dD_{\mathrm{KL}}/dt$",
        fontweight="bold"
    )

    axes[0].set_title(
        "KL Dissipation",
        fontweight="bold",
        pad=12
    )

    axes[1].set_ylabel(
        r"$\lambda(t)=-\frac{1}{D_{\mathrm{KL}}}\frac{dD_{\mathrm{KL}}}{dt}$",
        fontweight="bold"
    )

    axes[1].set_xlabel(
        "Time",
        fontweight="bold"
    )

    axes[1].set_title(
        "Relative KL Decay Rate",
        fontweight="bold",
        pad=12
    )

    for ax in axes:
        format_axis(ax)

    add_pair_legend(
        axes[0],
        pair_names
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            output_dir,
            "02_KL_dissipation.png"
        ),
        dpi=400,
        bbox_inches="tight"
    )

    plt.close(fig)


# ============================================================
# FIGURE 3: DIFFERENTIAL ENTROPY
# ============================================================

def plot_entropy(results, flow, output_dir):

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(13, 12),
        sharex=True
    )

    pair_names = sorted(results.keys())

    for index, pair_name in enumerate(pair_names):

        d = results[pair_name]

        color, linestyle, marker = pair_style(index)

        # Absolute entropy
        axes[0].plot(
            d["time"],
            d["entropy"],
            color=color,
            linestyle=linestyle,
            linewidth=2.5,
            marker=marker,
            markevery=25,
            markersize=5,
            label=pair_name.replace("_", " ")
        )

        # Entropy difference
        axes[1].plot(
            d["time"],
            d["delta_entropy"],
            color=color,
            linestyle=linestyle,
            linewidth=2.5,
            marker=marker,
            markevery=25,
            markersize=5,
            label=pair_name.replace("_", " ")
        )

    # Reference entropy varies by pair.
    # Show its range as a shaded band, rather than one incorrect
    # common reference line.
    ref_values = np.array([
        results[p]["reference_entropy"]
        for p in pair_names
    ])

    axes[0].axhspan(
        np.min(ref_values),
        np.max(ref_values),
        alpha=0.15,
        label="Range of reference entropies"
    )

    axes[0].set_ylabel(
        r"$H(t)$",
        fontweight="bold"
    )

    axes[0].set_title(
        "Differential Entropy",
        fontweight="bold",
        pad=12
    )

    axes[1].axhline(
        0,
        linewidth=2,
        linestyle="--"
    )

    axes[1].set_ylabel(
        r"$\Delta H(t)=H(t)-H_{\mathrm{ref}}$",
        fontweight="bold"
    )

    axes[1].set_xlabel(
        "Time",
        fontweight="bold"
    )

    axes[1].set_title(
        "Entropy Difference from Reference",
        fontweight="bold",
        pad=12
    )

    for ax in axes:
        format_axis(ax)

    add_pair_legend(
        axes[0],
        pair_names
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            output_dir,
            "03_differential_entropy.png"
        ),
        dpi=400,
        bbox_inches="tight"
    )

    plt.close(fig)


# ============================================================
# MAIN
# ============================================================

for flow in FLOWS:

    print(f"\nProcessing {flow}...")

    results = load_flow_data(flow)

    if len(results) == 0:
        print("No pair files found.")
        continue

    output_dir = os.path.join(
        OUTPUT_DIR,
        flow
    )

    os.makedirs(
        output_dir,
        exist_ok=True
    )

    # Exactly three figures per flow
    plot_redistribution(
        results,
        flow,
        output_dir
    )

    plot_kl_dissipation(
        results,
        flow,
        output_dir
    )

    plot_entropy(
        results,
        flow,
        output_dir
    )

    print(
        f"Completed {flow}: "
        f"{len(results)} pairs, 3 figures"
    )

print("\nAll publication figures generated.")


Processing linear...
Completed linear: 19 pairs, 3 figures

Processing trigonometric...
Completed trigonometric: 19 pairs, 3 figures

Processing spherical...
Completed spherical: 19 pairs, 3 figures

Processing operator...
Completed operator: 19 pairs, 3 figures

All publication figures generated.
